# 02 — Supervised Training: Siamese U-Net on OSCD

Train a Siamese U-Net for binary change detection on the OSCD100 dataset.

**Runtime**: Use Google Colab with T4 GPU (Runtime → Change runtime type → T4 GPU)

**Architecture**: Shared encoder for T1/T2, `|f_T1 - f_T2|` difference at each skip level, decoder outputs change mask.

**Loss**: Combined BCE + Dice (handles class imbalance).

In [ ]:
!git clone https://github.com/YOUR_USERNAME/delta.git 2>/dev/null || echo "Using local files"
!pip install torchgeo pytorch-lightning rasterio -q

In [ ]:
import os
import sys
import time
import json
import tempfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler
from torchgeo.datasets import OSCD100
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Setup: Clone repo and import modules

In [ ]:
if os.path.exists('delta/src'):
    sys.path.insert(0, 'delta')
    print("Using cloned repo")
elif os.path.exists('../src'):
    sys.path.insert(0, '..')
    print("Using parent dir")
else:
    print("Modules need to be available - clone the repo or upload files")

In [ ]:
from src.models.unet import UNet
from src.models.siamese_unet import SiameseUNet
from src.training.losses import CombinedLoss, compute_pos_weight
from src.training.metrics import evaluate_all
from src.data.ndvi import compute_ndvi_torch

print("All imports successful")

## 2. Dataset and DataLoader

In [ ]:
BATCH_SIZE = 8
PATCH_SIZE = 256
NUM_WORKERS = 2

root = os.path.join(tempfile.gettempdir(), 'oscd100')

train_dataset = OSCD100(root=root, split='train', bands=OSCD100.rgb_bands, download=True)
val_dataset = OSCD100(root=root, split='val', bands=OSCD100.rgb_bands, download=True)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
class OSCDWithNDVI(torch.utils.data.Dataset):
    def __init__(self, dataset, augment=False):
        self.dataset = dataset
        self.augment = augment

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image = sample['image'].float()
        mask = sample['mask'].float()

        if image.dim() == 4:
            t1, t2 = image[0], image[1]
        else:
            t1 = t2 = image

        t1 = self._add_ndvi(t1)
        t2 = self._add_ndvi(t2)

        if self.augment:
            if torch.rand(1).item() > 0.5:
                t1 = torch.flip(t1, [-1])
                t2 = torch.flip(t2, [-1])
                mask = torch.flip(mask, [-1])
            if torch.rand(1).item() > 0.5:
                t1 = torch.flip(t1, [-2])
                t2 = torch.flip(t2, [-2])
                mask = torch.flip(mask, [-2])

        image_stacked = torch.stack([t1, t2], dim=0)

        return {'image': image_stacked, 'mask': mask.unsqueeze(0)}

    def _add_ndvi(self, image):
        nir = image[0]
        red = image[2]
        ndvi = compute_ndvi_torch(nir, red)
        return torch.cat([image, ndvi.unsqueeze(0)], dim=0)


train_ds = OSCDWithNDVI(train_dataset, augment=True)
val_ds = OSCDWithNDVI(val_dataset, augment=False)

sample = train_ds[0]
print(f"Image shape: {sample['image'].shape}")
print(f"Mask shape: {sample['mask'].shape}")

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 3. Model, Loss, Optimizer

In [ ]:
model = SiameseUNet(
    in_channels=4,   # RGB + NDVI per date
    out_channels=1,
    features=[64, 128, 256, 512],
    use_attention=True,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

In [ ]:
all_labels = torch.cat([b['mask'] for b in train_loader], dim=0)
pos_weight = compute_pos_weight(all_labels).to(device)
print(f"Pos weight: {pos_weight.item():.2f}")

criterion = CombinedLoss(alpha=0.5, pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

print("Loss, optimizer, scheduler ready")

## 4. Training Loop

In [ ]:
MAX_EPOCHS = 80
EARLY_STOP_PATIENCE = 15
USE_AMP = device.type == 'cuda'
scaler = GradScaler() if USE_AMP else None

best_f1 = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_iou': []}

print(f"Training config:")
print(f"  Epochs: {MAX_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  LR: 1e-4")
print(f"  AMP: {USE_AMP}")
print(f"  Early stop patience: {EARLY_STOP_PATIENCE}")
print()

In [ ]:
for epoch in range(MAX_EPOCHS):
    t0 = time.time()

    model.train()
    train_loss = 0.0
    for batch in train_loader:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', enabled=USE_AMP):
            preds = model(images)
            loss = criterion(preds, masks)

        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    all_preds = []
    all_masks = []
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            preds = model(images)
            val_loss += criterion(preds, masks).item()
            all_preds.append(preds.cpu())
            all_masks.append(masks.cpu())

    val_loss /= len(val_loader)
    all_preds = torch.cat(all_preds)
    all_masks = torch.cat(all_masks)

    metrics_05 = evaluate_all(all_preds, all_masks, threshold=0.5)
    metrics_03 = evaluate_all(all_preds, all_masks, threshold=0.3)
    best_thresh_metrics = metrics_05 if metrics_05['f1'] > metrics_03['f1'] else metrics_03
    best_thresh = 0.5 if metrics_05['f1'] > metrics_03['f1'] else 0.3

    scheduler.step(best_thresh_metrics['f1'])

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(best_thresh_metrics['f1'])
    history['val_iou'].append(best_thresh_metrics['iou'])

    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch+1:3d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"F1@{best_thresh}: {best_thresh_metrics['f1']:.4f} | "
        f"IoU: {best_thresh_metrics['iou']:.4f} | "
        f"LR: {lr:.2e} | {elapsed:.1f}s"
    )

    if best_thresh_metrics['f1'] > best_f1:
        best_f1 = best_thresh_metrics['f1']
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_f1': best_f1,
            'config': {
                'model': {'type': 'siamese_unet', 'in_channels': 4, 'out_channels': 1,
                          'features': [64, 128, 256, 512], 'use_attention': True},
                'data': {'bands': ['B02', 'B03', 'B04', 'B08'], 'use_ndvi': True},
                'inference': {'overlap': 32, 'change_threshold': 0.15},
            },
        }, 'siamese_unet_best.pt')
        print(f"  -> Saved best model (F1: {best_f1:.4f})")
    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nTraining complete. Best F1: {best_f1:.4f}")

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs, history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['val_f1'], label='F1 Score', linewidth=2, color='tab:blue')
axes[1].plot(epochs, history['val_iou'], label='IoU', linewidth=2, color='tab:orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Siamese U-Net Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_supervised.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Visual Predictions

In [ ]:
model.eval()
n_show = min(4, len(val_ds))
fig, axes = plt.subplots(n_show, 4, figsize=(20, 5*n_show))

for i in range(n_show):
    sample = val_ds[i]
    image = sample['image'].unsqueeze(0).to(device)
    mask_gt = sample['mask'].squeeze().numpy()

    with torch.no_grad():
        pred = torch.sigmoid(model(image)).squeeze().cpu().numpy()

    pred_binary = (pred > 0.5).astype(np.uint8)

    t1 = image[0, 0, :3].cpu().numpy().transpose(1, 2, 0)
    t1 = np.clip(t1 / (t1.max() + 1e-10), 0, 1)
    t2 = image[0, 1, :3].cpu().numpy().transpose(1, 2, 0)
    t2 = np.clip(t2 / (t2.max() + 1e-10), 0, 1)

    axes[i, 0].imshow(t1)
    axes[i, 0].set_title('Before (T1)')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(t2)
    axes[i, 1].set_title('After (T2)')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(mask_gt, cmap='gray')
    axes[i, 2].set_title('Ground Truth')
    axes[i, 2].axis('off')

    axes[i, 3].imshow(pred_binary, cmap='gray')
    axes[i, 3].set_title(f'Prediction (F1: {evaluate_all(torch.from_numpy(pred).unsqueeze(0), torch.from_numpy(mask_gt).unsqueeze(0), 0.5)["f1"]:.3f})')
    axes[i, 3].axis('off')

plt.suptitle('Supervised Siamese U-Net Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('predictions_supervised.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Final Metrics

In [ ]:
for threshold in [0.3, 0.5, 0.7]:
    metrics = evaluate_all(all_preds, all_masks, threshold=threshold)
    print(f"Threshold {threshold}:")
    print(f"  F1: {metrics['f1']:.4f} | IoU: {metrics['iou']:.4f} | "
          f"Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f}")

## Summary

- Best model saved to `siamese_unet_best.pt`
- Training history saved in `history` variable
- Upload checkpoint to Google Drive for use in `04_patna_case_study.ipynb`
- Next: `03_train_self_supervised.ipynb` for the label-free approach